# YOLOv1 — PyTorch Implementation (From Scratch)
**Paper:** *You Only Look Once: Unified, Real-Time Object Detection* (CVPR 2016)

**Dataset:** Hocmay — bike, car, cat, cellphone, dog, person (6 classes)

**Pretrained:** ImageNet backbone (extraction.conv.weights)

**Setup:**
1. Upload `YOLOv1_Project.zip` to Google Drive root
2. `Runtime → Change runtime type → T4 GPU`
3. Run All

## Mount Drive & Extract Project

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, zipfile

PROJECT_ZIP = '/content/drive/MyDrive/YOLOv1_Project.zip'
PROJECT_DIR = '/content/yolov1'

if not os.path.exists(PROJECT_DIR):
    print('Extracting project...')
    with zipfile.ZipFile(PROJECT_ZIP, 'r') as z:
        z.extractall('/content/')
    for d in os.listdir('/content/'):
        full = f'/content/{d}'
        if os.path.isdir(full) and 'yolo' in d.lower() and d != 'yolov1':
            os.rename(full, PROJECT_DIR)
            break
    print(f'Extracted to: {PROJECT_DIR}')
else:
    print(f'Already exists: {PROJECT_DIR}')

# Fix slow matplotlib HSV → fast cv2
import re
with open(f'{PROJECT_DIR}/dataset.py', 'r') as f:
    code = f.read()
if 'matplotlib.colors' in code:
    old = """        import matplotlib.colors as mcolors
        img_float = img_np.astype(np.float32) / 255.0
        img_hsv_np = mcolors.rgb_to_hsv(img_float)  # values in [0, 1]

        # Saturation channel (index 1)
        sat_factor = random.uniform(1.0 / AUG_HSV_SATURATION, AUG_HSV_SATURATION)
        img_hsv_np[:, :, 1] = np.clip(img_hsv_np[:, :, 1] * sat_factor, 0, 1)

        # Value/exposure channel (index 2)
        exp_factor = random.uniform(1.0 / AUG_HSV_EXPOSURE, AUG_HSV_EXPOSURE)
        img_hsv_np[:, :, 2] = np.clip(img_hsv_np[:, :, 2] * exp_factor, 0, 1)

        img = Image.fromarray((mcolors.hsv_to_rgb(img_hsv_np) * 255).astype(np.uint8))"""
    new = """        import cv2
        img_bgr = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)
        img_hsv_cv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV).astype(np.float32)
        sat_factor = random.uniform(1.0 / AUG_HSV_SATURATION, AUG_HSV_SATURATION)
        img_hsv_cv[:, :, 1] = np.clip(img_hsv_cv[:, :, 1] * sat_factor, 0, 255)
        exp_factor = random.uniform(1.0 / AUG_HSV_EXPOSURE, AUG_HSV_EXPOSURE)
        img_hsv_cv[:, :, 2] = np.clip(img_hsv_cv[:, :, 2] * exp_factor, 0, 255)
        img_back = cv2.cvtColor(img_hsv_cv.astype(np.uint8), cv2.COLOR_HSV2BGR)
        img = Image.fromarray(cv2.cvtColor(img_back, cv2.COLOR_BGR2RGB))"""
    code = code.replace(old, new)
    with open(f'{PROJECT_DIR}/dataset.py', 'w') as f:
        f.write(code)
    print('HSV fix applied (cv2 — fast!)')
else:
    print('HSV already using cv2')

# Dataset is inside project zip
os.environ['YOLO_DATA_ROOT'] = f'{PROJECT_DIR}/data'
os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

import torch
print(f'\nWorking dir : {os.getcwd()}')
print(f'PyTorch     : {torch.__version__}')
if torch.cuda.is_available():
    print(f'GPU         : {torch.cuda.get_device_name(0)}')
else:
    print('WARNING: No GPU!')

## Verify Dataset & Architecture

In [ ]:
import os, torch
from config import TRAIN_IMG_DIR, VALID_IMG_DIR, TEST_IMG_DIR, CLASS_NAMES, DATA_ROOT, NUM_CLASSES, B, S
from model import YOLOv1

print('=== Dataset ===')
print(f'DATA_ROOT    : {DATA_ROOT}')
print(f'Classes ({NUM_CLASSES})  : {CLASS_NAMES}')
print(f'Train images : {len(os.listdir(TRAIN_IMG_DIR))}')
print(f'Valid images : {len(os.listdir(VALID_IMG_DIR))}')
print(f'Test  images : {len(os.listdir(TEST_IMG_DIR))}')

print('\n=== Architecture ===')
model = YOLOv1(num_classes=NUM_CLASSES, num_boxes=B, grid_size=S)
expected_out = S * S * (B * 5 + NUM_CLASSES)
print(f'Parameters   : {sum(p.numel() for p in model.parameters()):,}')
out = model(torch.randn(1, 3, 448, 448))
print(f'Output shape : {out.shape}  (expected: [1, {S}, {S}, {B*5+NUM_CLASSES}])')
assert out.shape == (1, S, S, B*5+NUM_CLASSES), f'Shape mismatch!'
print('Architecture OK!')

## Load Pretrained ImageNet Backbone

In [ ]:
import os
WEIGHTS_PATH = '/content/extraction.conv.weights'
if not os.path.exists(WEIGHTS_PATH):
    print('Downloading pretrained weights...')
    !wget -q --show-progress -O /content/extraction.conv.weights \
        https://github.com/frankzhangrui/Darknet-Yolo/raw/master/extraction.conv.weights
    print('Downloaded!')
else:
    print('Weights already downloaded')
print(f'Size: {os.path.getsize(WEIGHTS_PATH)/1e6:.1f} MB')

In [ ]:
import torch
from model import YOLOv1
from config import NUM_CLASSES, B, S
from pretrained import load_darknet_conv_weights

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

model = YOLOv1(num_classes=NUM_CLASSES, num_boxes=B, grid_size=S)
model = load_darknet_conv_weights(model, WEIGHTS_PATH, num_conv=20)
model = model.to(device)
print('\nModel ready with pretrained ImageNet backbone!')

## Visualize a Training Sample

In [ ]:
from dataset import YOLODataset
from config import TRAIN_IMG_DIR, TRAIN_LABEL_DIR
from visualize import visualize_sample
from IPython.display import display

ds = YOLODataset(TRAIN_IMG_DIR, TRAIN_LABEL_DIR, augment=False)
print(f'Dataset size: {len(ds)}')
img, target = ds[0]
print(f'Image: {img.shape}, Target: {target.shape}')
print(f'Objects: {(target[:,:,4]>0).sum().item()}')
vis_img = visualize_sample(ds, 0, '/content/sample_vis.jpg')
display(vis_img)

## Train YOLOv1

In [ ]:
import os, time, torch
from torch.utils.data import DataLoader
from config import (
    TRAIN_IMG_DIR, TRAIN_LABEL_DIR, VALID_IMG_DIR, VALID_LABEL_DIR,
    BATCH_SIZE, MOMENTUM, WEIGHT_DECAY, NUM_EPOCHS,
    LR_WARMUP_EPOCHS, LR_INIT, LR_MAIN, LR_SCHEDULE,
    CHECKPOINT_DIR, S, B, NUM_CLASSES,
)
from loss import YOLOv1Loss
from dataset import YOLODataset
from train import get_lr, train_one_epoch, validate

train_dataset = YOLODataset(TRAIN_IMG_DIR, TRAIN_LABEL_DIR, augment=True)
valid_dataset = YOLODataset(VALID_IMG_DIR, VALID_LABEL_DIR, augment=False)
print(f'Train: {len(train_dataset)}, Valid: {len(valid_dataset)}')

effective_batch = min(BATCH_SIZE, len(train_dataset))
train_loader = DataLoader(train_dataset, batch_size=effective_batch,
                          shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
valid_loader = DataLoader(valid_dataset, batch_size=effective_batch,
                          shuffle=False, num_workers=2, pin_memory=True)

criterion = YOLOv1Loss()
optimizer = torch.optim.SGD(model.parameters(), lr=LR_INIT,
                             momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f'Checkpoints: {CHECKPOINT_DIR}')

In [ ]:
best_mAP = 0.0
train_losses, val_losses, mAP_history = [], [], []

for epoch in range(NUM_EPOCHS):
    t_start = time.time()
    lr = get_lr(epoch)
    for pg in optimizer.param_groups:
        pg['lr'] = lr

    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device, epoch)
    train_losses.append(train_loss)

    val_loss, mAP, per_class_ap = 0.0, 0.0, {}
    if (epoch + 1) % 5 == 0 or epoch == NUM_EPOCHS - 1:
        val_loss, mAP, per_class_ap = validate(model, valid_loader, criterion, device)
        val_losses.append(val_loss)
        mAP_history.append((epoch + 1, mAP))

    elapsed = time.time() - t_start
    print(f'Epoch [{epoch+1}/{NUM_EPOCHS}]  LR: {lr:.6f}  '
          f'Train Loss: {train_loss:.4f}  Val Loss: {val_loss:.4f}  '
          f'mAP: {mAP:.4f}  Time: {elapsed:.1f}s')

    if per_class_ap:
        for cls_name, ap in per_class_ap.items():
            print(f'  {cls_name}: AP={ap:.4f}')

    if mAP > best_mAP:
        best_mAP = mAP
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'mAP': mAP, 'train_loss': train_loss},
                   os.path.join(CHECKPOINT_DIR, 'yolov1_best.pth'))
        print(f'  >> Saved best model (mAP={mAP:.4f})')

    if (epoch + 1) % 25 == 0:
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict()},
                   os.path.join(CHECKPOINT_DIR, f'yolov1_epoch{epoch+1}.pth'))

torch.save({'epoch': NUM_EPOCHS-1, 'model_state_dict': model.state_dict(), 'mAP': mAP},
           os.path.join(CHECKPOINT_DIR, 'yolov1_final.pth'))
print(f'\nTraining complete!  Best mAP: {best_mAP:.4f}')

## Plot Training Curves

In [ ]:
import matplotlib.pyplot as plt
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(range(1, len(train_losses)+1), train_losses, label='Train Loss')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Training Loss — Hocmay Dataset'); ax1.legend(); ax1.grid(True)
if mAP_history:
    ep, maps = zip(*mAP_history)
    ax2.plot(ep, maps, 'o-', color='green', label='mAP@0.5')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('mAP')
    ax2.set_title('Validation mAP@0.5'); ax2.legend(); ax2.grid(True)
plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=150)
plt.show()
print(f'Best mAP: {best_mAP:.4f}')

## Evaluate on Validation Set

In [ ]:
from evaluate import evaluate as full_evaluate
from config import CONF_THRESHOLD, NMS_THRESHOLD

best_path = os.path.join(CHECKPOINT_DIR, 'yolov1_best.pth')
ckpt = torch.load(best_path, map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
print(f'Loaded best checkpoint from epoch {ckpt["epoch"]+1}')

val_loss, mAP, per_class_ap = full_evaluate(
    model, valid_loader, criterion, device,
    conf_thresh=0.001, nms_thresh=NMS_THRESHOLD
)
print(f'\nValidation Results:')
print(f'  Loss    : {val_loss:.4f}')
print(f'  mAP@0.5 : {mAP:.4f}')
print(f'\nPer-class AP:')
for cls_name, ap in per_class_ap.items():
    print(f'  {cls_name:12s}: {ap:.4f}')

## Detection on Test Images

In [ ]:
import glob, matplotlib.pyplot as plt
from detect import detect_single
from config import TEST_IMG_DIR

# Use low conf_thresh=0.01 so weak predictions still show up
# (model is early-stage: mAP ~5%, scores are small but real)
DETECT_CONF = 0.01

test_images = sorted(glob.glob(os.path.join(TEST_IMG_DIR, '*.jpg')))[:6]
model.eval()
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()
for idx, img_path in enumerate(test_images):
    dets, result_img = detect_single(model, img_path, device, conf_thresh=DETECT_CONF)
    n_dets = dets.size(0) if dets.numel() > 0 else 0
    axes[idx].imshow(result_img)
    axes[idx].set_title(f'{os.path.basename(img_path)[:25]} | dets={n_dets} | conf>{DETECT_CONF}')
    axes[idx].axis('off')
plt.tight_layout()
plt.savefig('/content/test_detections.png', dpi=150)
plt.show()
print(f'Total test images shown: {len(test_images)}')

## Save Results to Drive

In [ ]:
import shutil
DRIVE_OUT = '/content/drive/MyDrive/YOLOv1_Results'
os.makedirs(DRIVE_OUT, exist_ok=True)
shutil.copytree(CHECKPOINT_DIR, f'{DRIVE_OUT}/checkpoints', dirs_exist_ok=True)
for f in ['training_curves.png', 'test_detections.png', 'sample_vis.jpg']:
    src = f'/content/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'{DRIVE_OUT}/{f}')
print(f'Saved to Drive: {DRIVE_OUT}')
print(f'Best mAP: {best_mAP:.4f}')